# Analysis of experiment #6: Pricing methods

## Read data

In [2]:
# Read csv

import pandas as pd
import numpy as np

df = pd.read_csv("stats6.csv")

# Add column with edge probability
df["p"] = df.instance.apply(lambda s: float(s.split("_")[2][1:]))

# Format timelimit
df.loc[df.state == 'TIME_EXCEEDED_LP', 'time'] = 900

# Ignore entries of solver "byp_p0" and "byp_p6"
df = df[df.solver != "byp_p0"]
df = df[df.solver != "byp_p6"]

print(list(df.columns))
df.head()

['instance', 'solver', 'run', 'nvertices', 'nedges', 'nP', 'nQ', 'nvars', 'ncons', 'state', 'terminationReason', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap', 'initialHeurValue', 'initialHeurTime', 'initialSemigreedyIters', 'nNodesInt', 'nNodesFrac', 'nNodesGcp', 'nNodesTrivial', 'nNodesInfeas', 'nNodesInfeasPrepro', 'nNodesInfeasCheck', 'nNodesInfeasAux', 'gcpAvgTime', 'nsol', 'nsolHeur', 'nsolLR', 'nsolGCP', 'nsolTrivial', 'ninitSol', 'ninitDummy', 'ninit', 'rootNVertices', 'rootNEdges', 'rootNP', 'rootNQ', 'rootlb', 'rootub', 'rootHeurTime', 'rootFeasTime', 'rootCgTime', 'rootNCalls', 'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1', 'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool', 'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact', 'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1', 'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime', 'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls', 'otherNodesNCallsPool', '

,instance,solver,run,nvertices,nedges,nP,nQ,nvars,ncons,state,...,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,p
0,r_n150_p0.25_nA30_nB45_i1,byp_p1,0,150,2822,30,45,-1,-1,OPTIMAL_RELAX,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25
1,r_n150_p0.5_nA30_nB30_i0,byp_p1,0,150,5548,30,30,-1,-1,TIME_EXCEEDED_LP,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50
2,r_n150_p0.25_nA30_nB45_i0,byp_p1,0,150,2747,30,45,-1,-1,TIME_EXCEEDED_LP,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25
3,r_n150_p0.5_nA30_nB45_i0,byp_p1,0,150,5548,30,45,-1,-1,OPTIMAL_RELAX,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50
4,r_n150_p0.25_nA45_nB45_i2,byp_p1,0,150,2800,45,45,-1,-1,TIME_EXCEEDED_LP,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25


## Summary

In [3]:
df3 = df.groupby(["solver"]).agg(
    {
        "state": [("solved", lambda x: (x != "TIME_EXCEEDED_LP").sum())],
        "rootTime": [("avg", "mean")],
        "rootTimeHeur": [("avg", "mean")],
        "rootTimeMwis1": [("avg", "mean")],
        "rootTimeMwis2": [("avg", "mean")],
        "rootTimeExact": [("avg", "mean")],
        "rootNCols": [("avg", "mean")],
        "rootNColsHeur": [("avg", "mean")],
        "rootNColsMwis1": [("avg", "mean")],
        "rootNColsMwis2": [("avg", "mean")],
        "rootNColsExact": [("avg", "mean")],
    }
).reset_index()
columns = [("solver",""), ("#solved",""), 
            ('pricing time (s)', 'total'), ('pricing time (s)', 'heur'), ('pricing time (s)', 'WSSPᴾ'), ('pricing time (s)', 'WSSPᴾꟴ'), ('pricing time (s)', 'exact'),
            ('#added cols', 'total'), ('#added cols', 'heur'), ('#added cols', 'WSSPᴾ'), ('#added cols', 'WSSPᴾꟴ'), ('#added cols', 'exact')]

df3.columns=pd.MultiIndex.from_tuples(columns)
df3[('pricing time (s)', 'total')] = df3[('pricing time (s)', 'total')].round(1)
df3[('pricing time (s)', 'heur')] = df3[('pricing time (s)', 'heur')].round(1)
df3[('pricing time (s)', 'WSSPᴾ')] = df3[('pricing time (s)', 'WSSPᴾ')].round(1)
df3[('pricing time (s)', 'WSSPᴾꟴ')] = df3[('pricing time (s)', 'WSSPᴾꟴ')].round(1)
df3[('pricing time (s)', 'exact')] = df3[('pricing time (s)', 'exact')].round(1)
df3[('#added cols', 'total')] = df3[('#added cols', 'total')].astype(int)
df3[('#added cols', 'heur')] = df3[('#added cols', 'heur')].astype(int)
df3[('#added cols', 'WSSPᴾ')] = df3[('#added cols', 'WSSPᴾ')].astype(int)
df3[('#added cols', 'WSSPᴾꟴ')] = df3[('#added cols', 'WSSPᴾꟴ')].round(1)
df3[('#added cols', 'exact')] = df3[('#added cols', 'exact')].astype(int)
df3 = df3.replace(0, "--")

# add initialization info
df3 = df3.drop(columns=[("solver","")])
df3

#solved pricing time (s)                          #added cols             \
                     total heur WSSPᴾ WSSPᴾꟴ  exact       total heur WSSPᴾ   
0      22            603.1   --    --     --  603.1         315   --    --   
1      28            449.2  1.2    --     --  448.0         551  512    --   
2      32            279.3  1.3   1.7     --  276.3         570  500    47   
3      29            418.4  1.2    --    2.7  414.5         554  512    --   
4      32            259.3  1.3   1.6    1.3  255.1         571  500    48   

                
  WSSPᴾꟴ exact  
0     --   315  
1     --    38  
2     --    21  
3    6.6    35  
4    1.5    20

In [4]:
df3.to_latex("table6.tex", index=False, float_format="%.1f", na_rep="--")